# Topic 9 — Vietnamese SMS Spam Detection
## Baseline có sẵn → Fine-tuning 1 (nhẹ) → Fine-tuning 2 (nâng cao)

**Mô hình:** `khanhthuan1995/autotrain-sms-spam-vietnamese-51799122495` (PhoBERT/RoBERTa đã học phân loại SMS Ham/Spam).

**Thiết kế thí nghiệm:**
1. **Baseline:** nạp checkpoint đã có, chỉ suy luận, **không training**.
2. **Fine-tuning 1:** dùng **cùng mô hình baseline**, đóng băng encoder, chỉ huấn luyện `classifier` **2 epochs**, AdamW cơ bản.
3. **Fine-tuning 2:** **tiếp tục từ trọng số sau FT1**, mở khóa toàn bộ PhoBERT, huấn luyện tối đa **3 epochs**, thêm các kỹ thuật tối ưu.
4. So sánh Accuracy, Precision/Recall/F1 Spam, Macro-F1, Confusion Matrix trên **cùng tập Test**.

**Đối với interview:** Các cell đánh số theo thứ tự; có thể chỉ trực tiếp Forward → Loss → Backward → Optimizer Step.

**Cách chạy:** Mở trong VS Code, chọn kernel `.venv` có PyTorch CUDA, để `vi_dataset.csv` cùng thư mục notebook. Không cần chạy file `.py` khác.

**Cài thư viện nếu thiếu — thực hiện một lần trong Terminal VS Code (không phải Markdown cell):**
```powershell
.\.venv\Scripts\python.exe -m pip install transformers huggingface_hub safetensors underthesea pandas numpy scikit-learn matplotlib ipykernel tqdm
```
PyTorch CUDA đã được cài riêng trong `.venv`. Nếu dùng Google Colab, thay đường dẫn dataset/kernels tương ứng.

**Lưu ý:** Baseline là checkpoint đã train SMS khác (theo mô tả của repository), không phải PhoBERT trần. Nguồn dữ liệu pretrained có thể trùng với tập đang dùng; chưa đủ bằng chứng loại trừ overlap.

## 1. Import và cấu hình
**Interview:** Batch 4 × tích lũy 4 bước ≈ effective batch 16 cho giai đoạn 2; FP16 và gradient checkpointing tiết kiệm VRAM của RTX 4050 6GB.

In [ ]:
import copy
import random
import re
import unicodedata
from pathlib import Path
from math import ceil

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

SEED = 42
CSV_PATH = Path("vi_dataset.csv")
MODEL_ID = "khanhthuan1995/autotrain-sms-spam-vietnamese-51799122495"
MAX_LENGTH = 128
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
ACCUM_STEPS = 4
FT1_EPOCHS = 2
FT2_EPOCHS = 3
FT1_LR = 1e-4
FT2_LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa hoạt động. Chọn kernel .venv đã cài PyTorch CUDA trong VS Code.")
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print(f"FT1={FT1_EPOCHS} epochs | FT2={FT2_EPOCHS} epochs | Batch={BATCH_SIZE} x Accum={ACCUM_STEPS}")

## 2. Tiền xử lý dữ liệu và chia Train / Validation / Test
**Interview:** Bỏ tin nhắn rỗng, loại nội dung trùng và các văn bản có nhãn mâu thuẫn **trước khi chia**. Giữ tỷ lệ Ham/Spam và dùng seed 42 để so sánh công bằng.

**Nhãn:** `ham`/`0` → Ham; `spam`/`1` → Spam. Nhãn `1` ở CSV cần được giảng viên/nguồn dữ liệu xác nhận nếu có khác biệt.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(value))).strip()

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Không thấy {CSV_PATH}. Thư mục làm việc: {Path.cwd()}")
raw = pd.read_csv(CSV_PATH, dtype=str)
if not {"labels", "texts_vi"}.issubset(raw.columns):
    raise ValueError("Dataset phải có 2 cột: labels, texts_vi")
raw["text"] = raw["texts_vi"].map(clean_text)
raw["label"] = raw["labels"].fillna("").str.strip().str.lower().map(
    {"ham": 0, "0": 0, "spam": 1, "1": 1}
)
if raw["label"].isna().any():
    raise ValueError("Có nhãn chưa được ánh xạ; kiểm tra cột labels trước khi train.")

not_empty = raw.loc[raw.text.ne("")].copy()
not_empty["key"] = not_empty["text"].str.casefold()
conflict = not_empty.groupby("key")["label"].nunique()
conflict_keys = set(conflict.index[conflict > 1])
df = (not_empty.loc[~not_empty.key.isin(conflict_keys)]
      .drop_duplicates(subset="key", keep="first")[["text", "label"]]
      .reset_index(drop=True))
df["label"] = df["label"].astype(int)

train_df, remaining = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(remaining, test_size=0.50, stratify=remaining.label, random_state=SEED)
train_df, val_df, test_df = (x.reset_index(drop=True) for x in (train_df, val_df, test_df))
for first, second in ((train_df, val_df), (train_df, test_df), (val_df, test_df)):
    assert set(first.text.str.casefold()).isdisjoint(set(second.text.str.casefold()))

summary = pd.DataFrame([
    {"Tập": name, "Tổng": len(frame), "Ham": int((frame.label == 0).sum()),
     "Spam": int((frame.label == 1).sum())}
    for name, frame in (("Train", train_df), ("Validation", val_df), ("Test", test_df))
])
display(summary)
print("Sau làm sạch:", len(df), "| Số văn bản mâu thuẫn nhãn:", len(conflict_keys))

## 3. Tải checkpoint và tokenize
**Interview:** Cùng tokenizer, preprocessing và một đối tượng `model` duy nhất được dùng xuyên suốt. `strict=True` bảo đảm classifier không bị khởi tạo ngẫu nhiên vì thiếu trọng số.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer, RobertaForSequenceClassification
from underthesea import word_tokenize

snapshot = Path(snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=["config.json", "pytorch_model.bin", "model.safetensors",
                    "vocab.txt", "bpe.codes", "tokenizer_config.json",
                    "special_tokens_map.json", "added_tokens.json"]
))
config = AutoConfig.from_pretrained(snapshot, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(snapshot, use_fast=False, local_files_only=True)
labels = {int(k): str(v).lower() for k, v in config.id2label.items()}
if config.num_labels != 2 or "spam" not in labels[1] or "spam" in labels[0]:
    raise ValueError(f"Mapping nhãn checkpoint khác 0=Ham, 1=Spam: {labels}")

# Nạp đúng trọng số encoder + classifier đã học.
model = RobertaForSequenceClassification(config)
safe_file = snapshot / "model.safetensors"
if safe_file.exists():
    from safetensors.torch import load_file
    weights = load_file(str(safe_file), device="cpu")
else:
    weights_file = snapshot / "pytorch_model.bin"
    if not weights_file.exists():
        raise FileNotFoundError("Thiếu trọng số model.safetensors hoặc pytorch_model.bin")
    weights = torch.load(weights_file, map_location="cpu", weights_only=True)
# position_ids là buffer, không phải trọng số train.
weights = {k: v for k, v in weights.items() if not k.endswith("embeddings.position_ids")}
model.load_state_dict(weights, strict=True)
del weights
model = model.to(device)
print("Checkpoint đã nạp đủ trọng số. Nhãn:", labels)

def build_tensor_dataset(frame):
    texts = [word_tokenize(t, format="text") for t in frame.text.tolist()]
    tokens = tokenizer(texts, padding="max_length", truncation=True,
                       max_length=MAX_LENGTH, return_tensors="pt")
    return TensorDataset(tokens["input_ids"], tokens["attention_mask"],
                         torch.tensor(frame.label.tolist(), dtype=torch.long))

train_set = build_tensor_dataset(train_df)
val_set = build_tensor_dataset(val_df)
test_set = build_tensor_dataset(test_df)
val_loader = DataLoader(val_set, batch_size=EVAL_BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=EVAL_BATCH_SIZE)

def make_train_loader():
    # Đặt lại seed để dễ tái lập thứ tự training.
    return DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))

## 4. Hàm đánh giá chung cho cả 3 giai đoạn
**Interview:** `model.eval()` và `torch.inference_mode()` chỉ dự đoán, không cập nhật trọng số. Dữ liệu mất cân bằng nên quan tâm **F1 Spam** bên cạnh Accuracy.

In [ ]:
def evaluate(model, loader):
    model.eval()
    true_labels, predicted_labels = [], []
    with torch.inference_mode():
        for ids, mask, labels_batch in loader:
            logits = model(input_ids=ids.to(device),
                           attention_mask=mask.to(device)).logits
            predicted_labels.extend(logits.argmax(dim=1).cpu().tolist())
            true_labels.extend(labels_batch.tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=[0, 1], zero_division=0)
    metrics = {
        "Accuracy": accuracy_score(true_labels, predicted_labels),
        "Precision Spam": float(precision[1]),
        "Recall Spam": float(recall[1]),
        "F1 Spam": float(f1[1]),
        "Macro-F1": float(f1.mean()),
    }
    return metrics, confusion_matrix(true_labels, predicted_labels, labels=[0, 1])

def show_result(name, metrics, cm):
    print("\n" + name)
    display(pd.DataFrame([metrics], index=[name]).mul(100).round(2))
    print("Confusion Matrix (thực tế là hàng Ham/Spam):\n", cm)

## 5. BASELINE — checkpoint có sẵn, KHÔNG training
**Interview:** Đây mới là baseline chính thức. Không tạo optimizer, không gọi `.backward()` hay `.step()`. Chỉ đo kết quả để so sánh.

In [ ]:
baseline_scores, baseline_cm = evaluate(model, test_loader)
show_result("Baseline (không training)", baseline_scores, baseline_cm)

## 6. FINE-TUNING 1 — nhẹ: chỉ training Classification Head (2 epochs)

- **Đóng băng** `model.roberta` bằng `requires_grad=False`.
- **Chỉ cập nhật** `model.classifier` bằng AdamW cơ bản, LR `1e-4`.
- **Không dùng** class weights, scheduler, early stopping hoặc grid search ở giai đoạn này.
- Chọn head tốt nhất bằng **Validation F1 Spam**, rồi đánh giá Test một lần.
- **Không load lại checkpoint**: sử dụng đúng `model` đã chạy baseline.

In [ ]:
# A1. Đóng băng encoder; classifier vẫn được cập nhật.
for parameter in model.roberta.parameters():
    parameter.requires_grad = False
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

# A2. Optimizer cơ bản bắt buộc phải có để mô hình học.
optimizer_head = torch.optim.AdamW(model.classifier.parameters(), lr=FT1_LR)
train_loader = make_train_loader()
best_head_score = -1.0
best_head_state = None
ft1_history = []

for epoch in range(1, FT1_EPOCHS + 1):
    model.train()
    model.roberta.eval()   # Encoder bị đóng băng; tắt dropout của encoder.
    running_loss = 0.0
    bar = tqdm(train_loader, desc=f"Fine-tuning 1 · Epoch {epoch}/{FT1_EPOCHS}")
    for ids, mask, y in bar:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optimizer_head.zero_grad(set_to_none=True)

        # A3. Forward pass: PhoBERT -> logits Ham/Spam.
        logits = model(input_ids=ids, attention_mask=mask).logits
        # A4. Tính Cross Entropy Loss cơ bản.
        loss = F.cross_entropy(logits.float(), y)
        # A5. Backward: chỉ classifier có gradient.
        loss.backward()
        # A6. AdamW cập nhật classification head.
        optimizer_head.step()
        running_loss += loss.item()

    # A7. Validation dùng để chọn head tốt nhất.
    val_scores, _ = evaluate(model, val_loader)
    ft1_history.append({"Epoch": epoch, "Training Loss": running_loss / len(train_loader),
                        "Validation F1 Spam": val_scores["F1 Spam"]})
    print(f"FT1 · Epoch {epoch}: Loss={running_loss / len(train_loader):.4f} | "
          f"Val F1 Spam={val_scores['F1 Spam']:.4f}")
    if val_scores["F1 Spam"] > best_head_score:
        best_head_score = val_scores["F1 Spam"]
        best_head_epoch = epoch
        # Chỉ lưu head (nhỏ) vào RAM, không sinh file mới.
        best_head_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.classifier.state_dict().items()})

model.classifier.load_state_dict(best_head_state, strict=True)
del best_head_state, optimizer_head
ft1_scores, ft1_cm = evaluate(model, test_loader)
show_result(f"Fine-tuning 1 (best epoch {best_head_epoch})", ft1_scores, ft1_cm)
display(pd.DataFrame(ft1_history).round(4))

## 7. FINE-TUNING 2 — nâng cao: mở khóa toàn bộ PhoBERT (3 epochs)

**Bắt đầu từ đúng trọng số đã học ở FT1:** không có lệnh tải lại `from_pretrained`, không khởi tạo classifier mới.

1. **Unfreeze Encoder:** mở cập nhật toàn bộ mô hình.
2. **Weighted Cross Entropy:** chú trọng lớp Spam ít mẫu hơn.
3. **AdamW + Weight Decay:** tối ưu trọng số.
4. **Warmup + Scheduler:** điều chỉnh tốc độ học.
5. **FP16 + Gradient Accumulation:** tiết kiệm VRAM 6 GB.
6. **Gradient Clipping:** hạn chế gradient quá lớn.
7. **Validation + Early Stopping:** chọn epoch tốt nhất trước khi Test.

**Khi interview:** chỉ thẳng dòng `scaler.scale(loss / group_size).backward()` và `scaler.step(optimizer)`.

In [ ]:
from transformers import get_linear_schedule_with_warmup

# B1. Mở khóa toàn bộ encoder (head vẫn tiếp tục được train).
for parameter in model.roberta.parameters():
    parameter.requires_grad = True
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

# B2. Dữ liệu training và Class Weights.
train_loader = make_train_loader()
counts = np.bincount(train_df.label, minlength=2)
if np.any(counts == 0):
    raise ValueError("Tập Train thiếu một trong hai lớp Ham/Spam")
class_weights = torch.tensor(len(train_df) / (2.0 * counts),
                             dtype=torch.float32, device=device)

# B3. Tối ưu nâng cao: AdamW, scheduler (warmup 10%), FP16.
optimizer = torch.optim.AdamW(model.parameters(), lr=FT2_LR, weight_decay=WEIGHT_DECAY)
total_updates = ceil(len(train_loader) / ACCUM_STEPS) * FT2_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, round(0.10 * total_updates)),
    num_training_steps=total_updates)
scaler = torch.amp.GradScaler("cuda")

ft2_history = []
best_ft2_f1 = -1.0
best_ft2_epoch = 0
best_ft2_state = None
no_improvement = 0

for epoch in range(1, FT2_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    num_batches = len(train_loader)
    bar = tqdm(enumerate(train_loader, start=1), total=num_batches,
               desc=f"Fine-tuning 2 · Epoch {epoch}/{FT2_EPOCHS}")

    for step, (ids, mask, y) in bar:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)

        # B4. Forward & Weighted Cross Entropy trong autocast FP16.
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits.float(), y, weight=class_weights)
        running_loss += loss.detach().item()

        # B5. Backward và tích lũy gradient: 4 batch × 4 mẫu.
        group_start = ((step - 1) // ACCUM_STEPS) * ACCUM_STEPS
        group_size = min(ACCUM_STEPS, num_batches - group_start)
        scaler.scale(loss / group_size).backward()

        if step % ACCUM_STEPS == 0 or step == num_batches:
            # B6. Gradient clipping -> Optimizer -> Scheduler -> Zero grad.
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= scale_before:   # Tránh step scheduler nếu overflow.
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    # B7. Đánh giá Validation, Early Stopping bằng F1 Spam.
    val_scores, _ = evaluate(model, val_loader)
    avg_loss = running_loss / num_batches
    ft2_history.append({"Epoch": epoch, "Training Loss": avg_loss,
                        "Validation F1 Spam": val_scores["F1 Spam"]})
    print(f"FT2 · Epoch {epoch}: Loss={avg_loss:.4f} | Val F1 Spam={val_scores['F1 Spam']:.4f}")
    if val_scores["F1 Spam"] > best_ft2_f1 + 1e-7:
        best_ft2_f1, best_ft2_epoch = val_scores["F1 Spam"], epoch
        no_improvement = 0
        # Lưu trong RAM CPU, không tạo file checkpoint ngoài.
        best_ft2_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE:
            print("Early Stopping: Validation F1 Spam không cải thiện trong", PATIENCE, "epochs")
            break

# B8. Khôi phục mô hình tốt nhất sau FT2.
model.load_state_dict(best_ft2_state, strict=True)
del best_ft2_state, optimizer, scheduler, scaler
ft2_scores, ft2_cm = evaluate(model, test_loader)
show_result(f"Fine-tuning 2 (best epoch {best_ft2_epoch})", ft2_scores, ft2_cm)
display(pd.DataFrame(ft2_history).round(4))

## 8. So sánh ba kết quả trực tiếp trong notebook
**Interview:** Baseline = 0 bước cập nhật; FT1 = huấn luyện head; FT2 = fine-tuning toàn bộ. Chỉ khác quy trình training, tập Test giữ nguyên.

**Cẩn trọng:** Nếu FT2 có điểm cao hơn FT1, đó là kết quả **cả một chiến lược** (unfreeze + class weights + scheduler + thêm epoch), không thể quy riêng cho một kỹ thuật.

In [ ]:
results = {"Baseline": baseline_scores,
           "Fine-tuning 1 (head only)": ft1_scores,
           "Fine-tuning 2 (full model)": ft2_scores}
matrices = {"Baseline": baseline_cm,
            "Fine-tuning 1": ft1_cm,
            "Fine-tuning 2": ft2_cm}
comparison = pd.DataFrame.from_dict(results, orient="index")
comparison.index.name = "Giai đoạn"
print("SO SÁNH KẾT QUẢ TEST (%):")
display((comparison * 100).round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (stage, cm) in zip(axes, matrices.items()):
    ax.imshow(cm, cmap="Blues")
    for (r, c), count in np.ndenumerate(cm):
        ax.text(c, r, str(count), ha="center", va="center",
                color="white" if count > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["Ham", "Spam"])
    ax.set_yticks([0, 1], ["Ham", "Spam"])
    ax.set_title(stage)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Nhãn thật")
plt.tight_layout()
plt.show()

print("Chênh lệch F1 Spam FT2 - FT1:",
      f"{(ft2_scores['F1 Spam']-ft1_scores['F1 Spam'])*100:+.2f} điểm phần trăm")

## 9. Demo thử tin nhắn mới (tùy chọn)
**Interview:** Dùng chính `model` sau FT2; không train thêm. Lưu ý xác suất softmax chưa được calibration, không phải tỷ lệ chính xác bảo đảm.

In [ ]:
@torch.inference_mode()
def predict_sms(text):
    model.eval()
    normalized = clean_text(text)
    if not normalized:
        raise ValueError("Nội dung SMS không được rỗng")
    segmented = word_tokenize(normalized, format="text")
    batch = tokenizer(segmented, truncation=True, max_length=MAX_LENGTH,
                      return_tensors="pt").to(device)
    logits = model(**batch).logits
    probs = torch.softmax(logits.float(), dim=1)[0].cpu().numpy()
    print("Dự đoán:", "Spam" if probs.argmax() == 1 else "Ham")
    print(f"P(Ham)={probs[0]:.2%} | P(Spam)={probs[1]:.2%}")

predict_sms("Chúc mừng! Bạn nhận được quà, vui lòng truy cập đường link để xác nhận.")

### Ghi chú khi nộp và vấn đáp

- Đây là **2 lần huấn luyện tiếp nối sau baseline không training**. FT1 chỉ tối ưu classifier; FT2 tiếp tục từ FT1 và tối ưu toàn mô hình.
- **AdamW ở FT1 vẫn là một optimizer**, vì đã có training thì cần cập nhật trọng số. Không dùng thuật toán nâng cao ở FT1 không có nghĩa là không dùng optimizer.
- Không xem Accuracy là đủ khi Ham nhiều hơn Spam; quan tâm Recall, F1 Spam và Macro-F1.
- Không khẳng định checkpoint gốc không trùng tập Test khi chưa xác minh nguồn dữ liệu huấn luyện ban đầu.
- Notebook **không tự sinh các file CSV, JSON, DOCX, ZIP hoặc checkpoint phụ**. Báo cáo Word riêng là tài liệu trình bày quy trình; sau khi chạy, hãy cập nhật metric thực tế vào báo cáo.
- Nếu máy thiếu VRAM 6 GB, có thể thử `BATCH_SIZE=2`, `ACCUM_STEPS=8` (giai đoạn 2).